In [ ]:
import os

# Defining Expected Columns
expected_columns = {
    "olist_customers_dataset": ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state", "_ingested_at"],
    "olist_products_dataset": ["product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm", "_ingested_at"],
    "olist_sellers_dataset": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state", "_ingested_at"]
}

# To handle drift, and decided to skip columns
ignored_columns = {
    "olist_customers_dataset": ["Unwanted_columns"],
    "olist_products_dataset": ["Unwanted_columns"],
    "olist_sellers_dataset": ["Unwanted_columns"]
}

drifted_tables = []

# Path
bronze_base_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"

# Schema Check Logic
dbutils.widgets.removeAll()
dbutils.widgets.text("table_name", "")
current_table = dbutils.widgets.get("table_name")

if current_table in expected_columns:
    print(f"--> Validating Schema for: {current_table}")

    # Loading the Delta table from Bronze
    try:
        df_bronze = spark.read.format("delta").load(os.path.join(bronze_base_path, current_table))

        current_cols = set(df_bronze.columns)
        expected_cols = set(expected_columns[current_table])
        ignored_cols = set(ignored_columns.get(current_table, []))

        #Identify Drift
        new_unhandled_cols = current_cols - expected_cols - ignored_cols
        missing_cols = expected_cols - current_cols

        if new_unhandled_cols or missing_cols:
            error_msg = f"Table: {current_table} | New: {new_unhandled_cols} | Missing: {missing_cols}"
            drifted_tables.append(error_msg)
            print(f"--> !! Schema Drift Detected: {error_msg}")
        else:
            print(f"--> Schema Check Passed for {current_table}")

    except Exception as e:
        print(f"--> !! Could not load Bronze table {current_table}. Error: {str(e)}")
        drifted_tables.append(f"{current_table}: Table load failed.")

# Checkpoint
if drifted_tables:
    report = "\n".join(drifted_tables)
    raise Exception(f"Validation Failed. Pipeline Halted:\n{report}")
else:
    print("--> Schema Validation successful. Starting Transformation...")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

# Parameters and Config
dbutils.widgets.removeAll()
dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

target_table = table_name

# Paths
bronze_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
silver_path = f"{silver_base}{target_table}"

# Source Load
df = spark.read.format("delta").load(f"{bronze_base}{table_name}")

# Transformation logic
def transform_silver(df, table):
    if table == "olist_customers_dataset":
        pk = "customer_id"
        df_clean = (df.select(
            F.trim(F.col("customer_id")).alias("customer_id"),
            F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
            "customer_zip_code_prefix",
            F.trim(F.col("customer_city")).alias("customer_city"),
            F.trim(F.col("customer_state")).alias("customer_state"),
            F.col("_ingested_at")))

    elif table == "olist_products_dataset":
        pk = "product_id"
        window_spec = Window.partitionBy("product_category_name")

        df_with_medians = (df.withColumn("m_weight", F.percentile_approx("product_weight_g", 0.5).over(window_spec))
                     .withColumn("m_length", F.percentile_approx("product_length_cm", 0.5).over(window_spec))
                     .withColumn("m_height", F.percentile_approx("product_height_cm", 0.5).over(window_spec))
                     .withColumn("m_width", F.percentile_approx("product_width_cm", 0.5).over(window_spec)))

        df_silver = (df_with_medians.select(
            F.trim(F.col("product_id")).alias("product_id"),
            F.coalesce(F.trim(F.col("product_category_name")), F.lit("outros")).alias("product_category_name"),
            F.col("product_name_lenght").cast("int").alias("product_name_length"),
            F.col("product_description_lenght").cast("int").alias("product_description_length"),
            F.col("product_photos_qty").cast("int").alias("product_photos_qty"),
            F.coalesce(F.col("product_weight_g").cast("int"), F.col("m_weight").cast("int")).alias("product_weight_g"),
            F.coalesce(F.col("product_length_cm").cast("int"), F.col("m_length").cast("int")).alias("product_length_cm"),
            F.coalesce(F.col("product_height_cm").cast("int"), F.col("m_height").cast("int")).alias("product_height_cm"),
            F.coalesce(F.col("product_width_cm").cast("int"), F.col("m_width").cast("int")).alias("product_width_cm"),
            F.col("_ingested_at")))

        df_clean = (df_silver.withColumn(
            "product_volume_cm3", 
            (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")).cast("int")))

    elif table == "olist_sellers_dataset":
        pk = "seller_id"
        df_clean = (df.select(
            F.trim(F.col("seller_id")).alias("seller_id"),
            "seller_zip_code_prefix",
            F.trim(F.col("seller_city")).alias("seller_city"),
            F.trim(F.col("seller_state")).alias("seller_state"),
            F.col("_ingested_at")))

    return df_clean, pk

df_transformed, primary_key = transform_silver(df, table_name)
pk_list = [primary_key] if isinstance(primary_key, str) else primary_key

# Deduplicating the incoming batch
df_final_batch = df_transformed.orderBy(F.col("_ingested_at").desc()).dropDuplicates(pk_list)

# Defining the "Beginning of Time" date for all historical data
beginning_of_time = F.lit("1900-01-01 00:00:00").cast("timestamp")

# Adding SCD Type 2 metadata
df_to_process = (df_final_batch
                 .withColumn("start_date", F.lit(beginning_of_time))
                 .withColumn("end_date", F.lit(None).cast("timestamp"))
                 .withColumn("is_current", F.lit(True))
                 .withColumn("silver_load_at", F.current_timestamp())
                 .drop("_ingested_at"))

# SCD Type 2 Execution Logic
if not DeltaTable.isDeltaTable(spark, silver_path):
    print(f"--> [INIT] Creating Silver Table: {table_name}")
    df_to_process.write.format("delta").mode("overwrite").save(silver_path)
else:
    print(f"--> Processing Silver Tables: {table_name}")
    dt_silver = DeltaTable.forPath(spark,silver_path)

    # Join on Natural key where target is the active record
    merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in pk_list]) + " AND target.is_current = true"

    # Excluding metadata columns from the comparison
    meta_cols = ["start_date", "end_date", "is_current", "silver_load_at"]
    data_cols = [c for c in df_to_process.columns if c not in meta_cols]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    # Updating existing records that changed
    (dt_silver.alias("target").merge(df_to_process.alias("source"), merge_condition)
     .whenMatchedUpdate(condition=change_condition, set={"end_date": "source.start_date", "is_current": "false"})
     .execute())

    # Appending new records
    df_to_insert = df_to_process.alias("source").join(
    spark.read.format("delta").load(silver_path).alias("target"),
    (F.col(f"source.{pk_list[0]}") == F.col(f"target.{pk_list[0]}")) & (F.col("target.is_current") == True),
    "left_anti")

    df_to_insert.write.format("delta").mode("append").save(silver_path)

# Audit & Exit
dt_final = DeltaTable.forPath(spark, silver_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")